FSGM attack

In [2]:
import torch
import torch.nn as nn


# fix seed so that random initialization always performs the same 
torch.manual_seed(13)


# create the model N as described in the question
N = nn.Sequential(nn.Linear(10, 10, bias=False),
                  nn.ReLU(),
                  nn.Linear(10, 10, bias=False),
                  nn.ReLU(),
                  nn.Linear(10, 3, bias=False))

# random input
x = torch.rand((1,10)) # the first dimension is the batch size; the following dimensions the actual dimension of the data
x.requires_grad_() # this is required so we can compute the gradient w.r.t x

t = 0 # target class

epsReal = 0.5  #depending on your data this might be large or small
eps = epsReal - 1e-7 # small constant to offset floating-point erros

# The network N classfies x as belonging to class 2
original_class = N(x).argmax(dim=1).item()  # TO LEARN: make sure you understand this expression
print("Original Class: ", original_class)
assert(original_class == 2)

# compute gradient
# note that CrossEntropyLoss() combines the cross-entropy loss and an implicit softmax function
L = nn.CrossEntropyLoss()
loss = L(N(x), torch.tensor([t], dtype=torch.long)) # TO LEARN: make sure you understand this line
loss.backward()

# your code here
# adv_x should be computed from x according to the fgsm-style perturbation such that the new class of xBar is the target class t above
# hint: you can compute the gradient of the loss w.r.t to x as x.grad
adv_x = x - eps * x.grad.sign()

new_class = N(adv_x).argmax(dim=1).item()
print("New Class: ", new_class)
assert(new_class == t)
# it is not enough that adv_x is classified as t. We also need to make sure it is 'close' to the original x. 
print(torch.norm((x-adv_x),  p=float('inf')).data)
assert( torch.norm((x-adv_x), p=float('inf')) <= epsReal)

Original Class:  2
New Class:  0
tensor(0.5000)


In [ ]:
# fix seed so that random initialization always performs the same 
import numpy as np

torch.manual_seed(13)


# create the model N as described in the question
N = nn.Sequential(nn.Linear(10, 10, bias=False),
                  nn.ReLU(),
                  nn.Linear(10, 10, bias=False),
                  nn.ReLU(),
                  nn.Linear(10, 3, bias=False))

# random input
x = torch.rand((1,10)) # the first dimension is the batch size; the following dimensions the actual dimension of the data
x.requires_grad_() # this is required so we can compute the gradient w.r.t x

t = 1 # target class

epsReal = 0.5 #depending on your data this might be large or small # cannot be greater than 9....... AWESOME!!!!!!
eps = epsReal - 1e-7 # small constant to offset floating-point errors


# The network N classfies x as belonging to class 2
original_class = N(x).argmax(dim=1).item()  # TO LEARN: make sure you understand this expression
print("Original Class: ", original_class)
assert(original_class == 2)



for alpha in np.arange(0.01 - 1e-7, eps, 0.01):

    new_class = original_class
    new_x = x
    index = 0

    # print("Alpha:", alpha)
    # print(new_x)

    while (new_class != t):

        if index == 100:
            # print("Too many iterations.")
            break

        # compute gradient
        # note that CrossEntropyLoss() combines the cross-entropy loss and an implicit softmax function
        L = nn.CrossEntropyLoss()
        loss = L(N(new_x), torch.tensor([t], dtype=torch.long)) # TO LEARN: make sure you understand this line
        loss.backward()

        # your code here
        # adv_x should be computed from x according to the fgsm-style perturbation such that the new class of xBar is the target class t above
        # hint: you can compute the gradient of the loss w.r.t to x as x.grad
        new_x = new_x - alpha * new_x.grad.sign()
        # new_x = torch.clamp(new_x, x - eps, x + eps)

        new_class = N(new_x).argmax(dim=1).item()
        index += 1

        new_x = new_x.detach()
        new_x.requires_grad_()
        # new_x.retain_grad()
        # print(new_x)
    
    if new_class == t:
        print("Got it with alpha", alpha)
        break

print("New Class: ", new_class)
assert(new_class == t)
# it is not enough that adv_x is classified as t. We also need to make sure it is 'close' to the original x. 
print(torch.norm((x-new_x),  p=float('inf')).data)
assert( torch.norm((x-new_x), p=float('inf')) <= epsReal)

print("Number of FGSM iterations: ", index)


Original Class:  2
Got it with alpha 0.2499999
New Class:  1
tensor(0.5000)
Number of FGSM iterations:  18
